# Zadanie - Kryminalna zagadka z LLM Reasoning

## Treść zagadki

W poniedziałek z firmowej lodówki zniknął ostatni kawałek tortu urodzinowego. Czworo pracowników jest podejrzanych: **Adam, Beata, Celina i Dawid**.

**Zasady:**
- Winowajca we WSZYSTKICH swoich zeznaniach mówi nieprawdę (zawsze kłamie).
- Osoby niewinne we WSZYSTKICH swoich zeznaniach mówią prawdę (zawsze mówią prawdę).
- Winowajca jest dokładnie jedna osoba.

**Fakty zebrane przez detektywa:**
- Wszyscy czworo pracowników zajmują wspólną, otwartą przestrzeń biurową (open space) na 3. piętrze.
- Na tym samym piętrze, przy końcu korytarza za open space, znajduje się kuchnia z lodówką — dostępna dla wszystkich pracowników. Wyjście z open space prowadzi właśnie na ten korytarz.
- Tort należał do Ewy. Przyniosła go rano z okazji swoich urodzin i umieściła w lodówce w kuchni. Brak tortu odkryła o 15:30, wracając z zewnętrznego spotkania.
- Kuchnia nie jest objęta monitoringiem.
- Monitoring przy wejściu do budynku potwierdza, że żaden z czterech podejrzanych nie opuścił budynku między 12:00 a 16:00.
- Sala konferencyjna na 2. piętrze była zarezerwowana od 14:00 do 15:30. Lista obecności ze spotkania liczy 8 osób.

**Zeznania:**

**Adam:**
1. „Widziałem Beatę w kuchni o 13:30."
2. „Widziałem Dawida wchodzącego na salę konferencyjną przed 14:00."
3. „Widziałem tort w lodówce gdy brałem wodę z kuchni o 12:50."

**Beata:**
1. „Widziałam, jak Celina wychodzi z open space przed 13:00."
2. „Adam jadł lunch obok mnie w kuchni o 13:30."

**Celina:**
1. „Nie ruszałam się z open space przed 13:00."
2. „Dawid o 14:00 kręcił się po korytarzu, nie był na żadnym spotkaniu."
3. „Widziałam tort w lodówce gdy wychodziłam po dokumenty chwilę po 13:00."

**Dawid:**
1. „Byłem na spotkaniu o 14:00."
2. „Beata rozmawiała ze mną przy wejściu do open space o 12:45."

---

> **Prawidłowe rozwiązanie:** winowajcą jest **Celina**.
>
> Pozostałe osoby można wykluczyć przez sprzeczność:
>
> - **Adam jako winowajca** — Adam kłamałby, więc Beata i Celina mówiłyby prawdę. Beata (prawda, B1) twierdzi, że widziała, jak Celina wychodzi przed 13:00, ale Celina (prawda, C1) twierdzi, że się nie ruszała — sprzeczność.
> - **Beata jako winowajca** — Beata kłamałaby, więc Adam i Celina mówiłyby prawdę. Adam (prawda, A2) twierdzi, że widział Dawida wchodzącego na salę konferencyjną, ale Celina (prawda, C2) twierdzi, że Dawid kręcił się po korytarzu i nie był na spotkaniu — sprzeczność.
> - **Dawid jako winowajca** — Dawid kłamałby, więc jego zeznanie D1 („Byłem na spotkaniu") byłoby fałszywe. Ale Adam (prawda, A2) twierdzi, że widział Dawida wchodzącego na salę konferencyjną — sprzeczność.
>
> **Celina jako winowajca** — wszystkie zeznania są spójne: Celina wyszła z open space przed 13:00 (B1 Beaty potwierdza jej kłamstwo C1), Dawid był na spotkaniu (D1 Dawida i A2 Adama są zgodne, C2 Celiny jest kłamstwem), tort był w lodówce o 12:50 (A3 Adama), a Celina zabrała go wychodząc przed 13:00 (jej zeznanie C3 o widzeniu tortu „chwilę po 13:00" jest kłamstwem).

## Rozwiązanie zagadki bez i z reasoningiem

Wyślij powyższą zagadkę do modelu **gpt-5.4** dwukrotnie, używając Responses API (`client.responses.create`):

1. Z parametrem `reasoning={"effort": "none"}` (bez reasoningu)
2. Z parametrem `reasoning={"effort": "high"}` (z wysokim reasoningiem)

Dla każdego z wywołań wyświetl:
- Treść odpowiedzi modelu

Porównaj wyniki: czy oba wywołania dają tę samą odpowiedź?

In [ ]:
from openai import OpenAI
import os

# Inicjalizacja klienta
api_key = os.getenv('OPENAI_API_KEY') # pobranie klucza API ze zmiennej środowiskowej
client = OpenAI(api_key=api_key)      # konfiguracja połączenia z API

# Treść zagadki - opis sytuacji, faktów i zeznań
zagadka = """
W poniedziałek z firmowej lodówki zniknął ostatni kawałek tortu urodzinowego.
Czworo pracowników jest podejrzanych: Adam, Beata, Celina i Dawid.

Zasady:
- Winowajca we WSZYSTKICH swoich zeznaniach mówi nieprawdę (zawsze kłamie).
- Osoby niewinne we WSZYSTKICH swoich zeznaniach mówią prawdę (zawsze mówią prawdę).
- Winowajca jest dokładnie jedna osoba.

Fakty zebrane przez detektywa:
- Wszyscy czworo pracowników zajmują wspólną, otwartą przestrzeń biurową (open space) na 3. piętrze.
- Na tym samym piętrze, przy końcu korytarza za open space, znajduje się kuchnia z lodówką - dostępna dla wszystkich pracowników. Wyjście z open space prowadzi właśnie na ten korytarz.
- Tort należał do Ewy. Przyniosła go rano z okazji swoich urodzin i umieściła w lodówce w kuchni. Brak tortu odkryła o 15:30, wracając z zewnętrznego spotkania.
- Kuchnia nie jest objęta monitoringiem.
- Monitoring przy wejściu do budynku potwierdza, że żaden z czterech podejrzanych nie opuścił budynku między 12:00 a 16:00.
- Sala konferencyjna na 2. piętrze była zarezerwowana od 14:00 do 15:30. Lista obecności ze spotkania liczy 8 osób.

Zeznania:

Adam:
  1. Widziałem Beatę w kuchni o 13:30.
  2. Widziałem Dawida wchodzącego na salę konferencyjną przed 14:00.
  3. Widziałem tort w lodówce gdy brałem wodę z kuchni o 12:50.

Beata:
  1. Widziałam, jak Celina wychodzi z open space przed 13:00.
  2. Adam jadł lunch obok mnie w kuchni o 13:30.

Celina:
  1. Nie ruszałam się z open space przed 13:00.
  2. Dawid o 14:00 kręcił się po korytarzu, nie był na żadnym spotkaniu.
  3. Widziałam tort w lodówce gdy wychodziłam po dokumenty chwilę po 13:00.

Dawid:
  1. Byłem na spotkaniu o 14:00.
  2. Beata rozmawiała ze mną przy wejściu do open space o 12:45.
"""

## Stretch: Podgląd procesu rozumowania i liczby wykorzystanych tokenów

Wyślij zagadkę jeszcze raz z parametrami:
- `reasoning={"effort": "high", "summary": "detailed"}`

Następnie:
1. Iteruj po `response.output` i oddzielnie wyświetl:
   - **Podsumowanie rozumowania** (elementy typu `"reasoning"` z polem `summary`)
   - **Finalną odpowiedź** (elementy typu `"message"`)
2. Wyświetl statystyki tokenów, w tym osobno liczbę `reasoning_tokens` i tokenów odpowiedzi.